# 04 — Near-optimality, route branches, values table, tiebreak

Fourth of four (spec §9, steps 4a–4d). Products: `near_optimality.tif` (**D11**, wall-to-wall
slack in raw cost units, **G10**), route branches (**D12**, **G9**) with the
`route_irreplaceable` flag, the per-branch values table on the Y2Y-wide column spec (**D13**,
**G11**) with the Carroll 2018 audit column (**D14**, H6-guarded), and the tie-break ranking.
Ends with `cc.finish(A)` — the completed run + `runs.csv` index.


In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/northern_connectivity/, below the repo root where config.py
# and the corridor engine modules (corridors_prep / corridor_graph / corridors_core /
# corridors_ensemble) sit. Same bootstrap pattern as analyses/y2y/.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
import corridors_ensemble as ce
for _m in (config, cp, cg, cc, ce):
    importlib.reload(_m)

KEY = "north"


In [ ]:
# ---- Re-attach to the run created by 02_calibrate_baseline ------------------
# One run spans notebooks 02-04: 02 created it via cc.start(); here cc.load() reads ONLY the run
# dir (run_config.json + the H7 copies), then the cached CWD + graph are re-derived -- the CWD
# stage is a cache HIT (seconds), the network rebuild is minutes. cwd_cutoff_abs was written into
# run_config.json by cc.set_cutoff in notebook 02, so nothing here depends on config.py.
RUN = "v2_run001"                       # <-- the run to continue

A = cc.load(config.RESULTS_DIR / "corridors_north" / RUN)
cc.resistance(A)
cc.cost_distances(A)                    # cache HIT
cc.corridor_network(A, verbose=False)
A


## Step 4a · Near-optimality surface (**D11**) · **gate G10**

`min_e slack_e` in raw cost units on every routable cell — the band IS the closed-form
near-optimal set and slack its continuous degree. Tiers are percentiles of slack over the union
band at 2× cutoff (pre-registered in config). Zero-cost adjacency edges contribute nothing.


In [ ]:
cc.near_optimality(A)


## Step 4b · Route branches (**D12**) · **gate G9**

Per edge (locked intra-name edges included), 8-connected band components at
`branch_mult × cutoff`, formed BEFORE node subtraction (G9 asserts each touches both
endpoints), then node land removed and slivers < `branch_min_km2` dropped (count reported).
`n_branches == 1` ⇒ route-irreplaceable — reported alongside, never merged with, the D7
edge-irreplaceable flag.


In [ ]:
cc.route_branches(A)


## Step 4c · Per-branch values table (**D13/D14**) · **gate G11**

Masks cross 300 m → 1 km (`_to_audit`, ≥ 0.5 areal fraction); columns are imported from
`results_core` (RAW_SPEC / `mask_profile`) — same spec as the Y2Y-wide alternatives tables.
Row unit is edge × route branch, NOT a solution cluster (the caption ships in
`alternatives_branches.meta.json`). `carroll2018_pctl` is audit-only (D14); if the layer is
absent the gap is logged, not fatal (H6).


In [ ]:
cc.alternatives_table(A)


## Step 4d · Tie-break report (Phase 8.3, small)

Ranking only — no automated "recommended" flag; the recommendation is a human read of the table.


In [ ]:
cc.tiebreak(A)


## Finish — complete the run + the analysis-level index


In [ ]:
cc.finish(A)
cc.runs(KEY)
